In [ ]:
import torch
import triton
import triton.language as tl
from triton.testing import do_bench

print(torch.__version__, triton.__version__, torch.cuda.get_device_name())

torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
@triton.autotune(
    configs=[
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32, "SPLIT_K": 1},
            num_warps=8,
            num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 128, "BLOCK_K": 32, "SPLIT_K": 1},
            num_warps=8,
            num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32, "SPLIT_K": 2},
            num_warps=8,
            num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 128, "BLOCK_K": 32, "SPLIT_K": 2},
            num_warps=8,
            num_stages=4,
        ),
    ],
    key=["M", "N", "K"],
    reset_to_zero=["c_ptr"],
)
@triton.jit
def matmul_kernel2(
    a_ptr,
    b_ptr,
    c_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    SPLIT_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    pid_k = tl.program_id(2)

    K_CHUNK = tl.cdiv(K, SPLIT_K)

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, K_CHUNK, BLOCK_K):
        am_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        am_mask = am_offsets < M
        ak_offsets = pid_k * K_CHUNK + k + tl.arange(0, BLOCK_K)
        ak_mask = ak_offsets < min((pid_k + 1) * K_CHUNK, K)

        a = tl.load(
            a_ptr + am_offsets[:, None] * stride_am + ak_offsets * stride_ak,
            mask=am_mask[:, None] & ak_mask,
            other=0.0,
        )

        bn_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        bn_mask = bn_offsets < N
        bk_offsets = pid_k * K_CHUNK + k + tl.arange(0, BLOCK_K)
        bk_mask = bk_offsets < min((pid_k + 1) * K_CHUNK, K)

        b = tl.load(
            b_ptr + bk_offsets[:, None] * stride_bk + bn_offsets * stride_bn,
            mask=bk_mask[:, None] & bn_mask,
            other=0.0,
        )

        acc = tl.dot(a, b, acc=acc)

    cm_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cm_mask = cm_offsets < M
    cn_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    cn_mask = cn_offsets < N

    tl.atomic_add(
        c_ptr + cm_offsets[:, None] * stride_cm + cn_offsets * stride_cn,
        acc,
        mask=cm_mask[:, None] & cn_mask,
    )


def matmul2(a, b, **kwargs):
    assert a.ndim == b.ndim == 2
    assert a.shape[1] == b.shape[0]
    assert a.device == b.device
    assert a.dtype == b.dtype

    M, K = a.shape
    _, N = b.shape

    out = torch.zeros((M, N), device=a.device, dtype=torch.float32)

    grid = lambda meta: (
        triton.cdiv(M, meta["BLOCK_M"]),
        triton.cdiv(N, meta["BLOCK_N"]),
        meta["SPLIT_K"],
    )

    matmul_kernel2[grid](
        a,
        b,
        out,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        out.stride(0),
        out.stride(1),
    )

    return out.to(a.dtype)

In [ ]:
print(f"{'k_size':8s} {'cublas':14s} {'yours':14s}")
for size in [
    16,
    32,
    64,
    128,
    256,
    512,
    1024,
    2048,
    4096,
    8192,
    16384,
]:  # beyond 64 out of shared memory
    a = torch.randn(1024, size, device="cuda", dtype=torch.float16)
    b = torch.randn(size, 1024, device="cuda", dtype=torch.float16)
    cublas = lambda: torch.matmul(a, b)
    yours = lambda: matmul2(a, b, BLOCK_K=size)

    torch.testing.assert_close(cublas(), yours(), atol=1e-1, rtol=1e-1)

    ms_cublas = do_bench(cublas, warmup=10, rep=100)
    ms_triton = do_bench(yours, warmup=10, rep=100)
    print(f"{size:<8d} {ms_cublas:<14.3f} {ms_triton:<14.3f}")

In [ ]:
print(f"{'mn_size':8s} {'cublas':14s} {'yours':14s}")
for size in [
    16,
    32,
    64,
    128,
    256,
    512,
    1024,
    2048,
    4096,
    8192,
    16384,
]:  # beyond 64 out of shared memory
    a = torch.randn(size, 1024, device="cuda", dtype=torch.float16)
    b = torch.randn(1024, size, device="cuda", dtype=torch.float16)
    cublas = lambda: torch.matmul(a, b)
    yours = lambda: matmul2(a, b, BLOCK_K=size)

    torch.testing.assert_close(cublas(), yours(), atol=1e-1, rtol=1e-1)

    ms_cublas = do_bench(cublas, warmup=10, rep=100)
    ms_triton = do_bench(yours, warmup=10, rep=100)
    print(f"{size:<8d} {ms_cublas:<14.3f} {ms_triton:<14.3f}")

In [ ]:
cublas()

In [ ]:
yours()